In [15]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv("feature_selected_dataset.csv")

# ensure proper format
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

# =========================================================
# TARGET COLUMN
# =========================================================
TARGET_COL = "target_next_return"

# optional sanity check
required_cols = ["date", "ticker", TARGET_COL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

In [17]:
# =========================================================
# TIME SPLIT
# train = up to 2023
# val   = 2024
# test  = 2025
# =========================================================
train_df = df[df["date"].dt.year <= 2023].copy()
val_df   = df[df["date"].dt.year == 2024].copy()
test_df  = df[df["date"].dt.year == 2025].copy()

print("Train:", train_df["date"].min(), "→", train_df["date"].max(), "| rows =", len(train_df))
print("Val:  ", val_df["date"].min(),   "→", val_df["date"].max(),   "| rows =", len(val_df))
print("Test: ", test_df["date"].min(),  "→", test_df["date"].max(),  "| rows =", len(test_df))


Train: 2005-02-01 00:00:00 → 2023-12-29 00:00:00 | rows = 22424
Val:   2024-01-02 00:00:00 → 2024-12-31 00:00:00 | rows = 1260
Test:  2025-01-02 00:00:00 → 2025-12-30 00:00:00 | rows = 1245


In [19]:
# =========================================================
# FEATURE LIST
# exclude identifiers + target columns
# VERY IMPORTANT: exclude target_next_price too
# =========================================================
exclude_cols = [
    "date",
    "ticker",
    "target_next_return",
    "target_next_price"
]

features = [col for col in df.columns if col not in exclude_cols]

print("\nNumber of features:", len(features))
print("Features:", features)


Number of features: 17
Features: ['close', 'volume', 'simple_return', 'return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_5', 'return_lag_10', 'volume_lag_1', 'volume_lag_5', 'roc_20', 'ewm_vol_10', 'price_to_sma_10', 'price_to_sma_20', 'relative_return_1', 'relative_strength_5', 'relative_strength_20']


In [21]:
# =========================================================
# SCALE FEATURES
# fit ONLY on train, transform val/test
# =========================================================
scaler = StandardScaler()

train_df.loc[:, features] = scaler.fit_transform(train_df[features])
val_df.loc[:, features]   = scaler.transform(val_df[features])
test_df.loc[:, features]  = scaler.transform(test_df[features])

/var/folders/_h/b34996m51sb595dtyp0k7l300000gn/T/ipykernel_13562/3392736179.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 8.33983109 15.01137812  5.54371965 ... -0.48910531 -0.46793134
 -0.4509838 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, features] = scaler.fit_transform(train_df[features])
/var/folders/_h/b34996m51sb595dtyp0k7l300000gn/T/ipykernel_13562/3392736179.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.37964887 -0.37365845 -0.33867028 ... -0.52037772 -0.53069981
 -0.51506898]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  val_df.loc[:, features]   = scaler.transform(val_df[features])
/var/folders/_h/b34996m51sb595dtyp0k7l300000gn/T/ipykernel_13562/3392736179.py:9: FutureWarning: Setting an item of incompatible d

In [23]:
# =========================================================
# SPLIT X / y
# =========================================================
X_train = train_df[features]
y_train = train_df[TARGET_COL]

X_val = val_df[features]
y_val = val_df[TARGET_COL]

X_test = test_df[features]
y_test = test_df[TARGET_COL]

In [25]:
# =========================================================
# LINEAR REGRESSION
# =========================================================
lr = LinearRegression()
lr.fit(X_train, y_train)

val_pred = lr.predict(X_val)
test_pred = lr.predict(X_test)

mse_lr_val = mean_squared_error(y_val, val_pred)
mse_lr_test = mean_squared_error(y_test, test_pred)

print("\nLR Validation MSE:", mse_lr_val)
print("LR Test MSE:", mse_lr_test)


LR Validation MSE: 0.0005019704484174227
LR Test MSE: 0.0005473118544540764


In [27]:
import pandas as pd
from sklearn.metrics import mean_squared_error

# =========================================================
# CONFIG
# =========================================================
TARGET_COL = "target_next_return"
window = 5

# ensure proper format
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

val_preds = []
test_preds = []

for ticker, group in df.groupby("ticker"):
    group = group.sort_values("date").copy()

    # moving-average forecast using past observed target values only
    # rolling(window).mean() at time t uses past window values up to t
    # shift(1) makes prediction for time t use info only up to t-1
    group["ma_pred"] = group[TARGET_COL].rolling(window=window).mean().shift(1)

    val_part = group[group["date"].dt.year == 2024].copy()
    test_part = group[group["date"].dt.year == 2025].copy()

    if not val_part.empty:
        val_preds.append(val_part)

    if not test_part.empty:
        test_preds.append(test_part)

val_ma_df = pd.concat(val_preds, ignore_index=True)
test_ma_df = pd.concat(test_preds, ignore_index=True)

# drop rows where rolling prediction is unavailable
val_ma_df = val_ma_df.dropna(subset=["ma_pred", TARGET_COL]).copy()
test_ma_df = test_ma_df.dropna(subset=["ma_pred", TARGET_COL]).copy()

mse_ma_val = mean_squared_error(val_ma_df[TARGET_COL], val_ma_df["ma_pred"])
mse_ma_test = mean_squared_error(test_ma_df[TARGET_COL], test_ma_df["ma_pred"])

print("MA Validation MSE:", mse_ma_val)
print("MA Test MSE:", mse_ma_test)

MA Validation MSE: 0.0006050247325426146
MA Test MSE: 0.0006782455027577103


In [29]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

# =========================================================
# CONFIG
# =========================================================
TARGET_COL = "target_next_return"
ARIMA_ORDER = (1, 0, 1)

# ensure proper format
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

arima_val_preds = []
arima_test_preds = []

for ticker, group in df.groupby("ticker"):
    group = group.sort_values("date").copy()

    train_series = group.loc[group["date"].dt.year <= 2023, TARGET_COL]
    val_series_df = group.loc[group["date"].dt.year == 2024].copy()
    test_series_df = group.loc[group["date"].dt.year == 2025].copy()

    # skip if any split is empty
    if train_series.empty or val_series_df.empty or test_series_df.empty:
        print(f"Skipping {ticker}: missing train/val/test data")
        continue

    try:
        # -------------------------------
        # Fit on train, forecast validation
        # -------------------------------
        model_val = ARIMA(train_series, order=ARIMA_ORDER)
        model_val_fit = model_val.fit()

        val_forecast = model_val_fit.forecast(steps=len(val_series_df))

        temp_val = val_series_df.copy()
        temp_val["arima_pred"] = val_forecast.values
        arima_val_preds.append(temp_val)

        # -------------------------------
        # Refit on train + validation, forecast test
        # -------------------------------
        train_val_series = group.loc[group["date"].dt.year <= 2024, TARGET_COL]

        model_test = ARIMA(train_val_series, order=ARIMA_ORDER)
        model_test_fit = model_test.fit()

        test_forecast = model_test_fit.forecast(steps=len(test_series_df))

        temp_test = test_series_df.copy()
        temp_test["arima_pred"] = test_forecast.values
        arima_test_preds.append(temp_test)

    except Exception as e:
        print(f"Skipping {ticker} due to ARIMA error: {e}")
        continue

arima_val_df = pd.concat(arima_val_preds, ignore_index=True)
arima_test_df = pd.concat(arima_test_preds, ignore_index=True)

arima_val_df = arima_val_df.dropna(subset=[TARGET_COL, "arima_pred"]).copy()
arima_test_df = arima_test_df.dropna(subset=[TARGET_COL, "arima_pred"]).copy()

mse_arima_val = mean_squared_error(arima_val_df[TARGET_COL], arima_val_df["arima_pred"])
mse_arima_test = mean_squared_error(arima_test_df[TARGET_COL], arima_test_df["arima_pred"])

print("ARIMA Validation MSE:", mse_arima_val)
print("ARIMA Test MSE:", mse_arima_test)

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next v

ARIMA Validation MSE: 0.000498883855131883
ARIMA Test MSE: 0.0005469370369025533


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
